# 实验四：神经压缩 vs JPEG（Neural Compression vs JPEG）
## FMI Course · Kaggle Hands-on Lab

**课程**：未来媒体互联网（Future Media & Internet）&nbsp;|&nbsp; **预计时长**：~10 分钟 &nbsp;|&nbsp; **运行环境**：Kaggle Notebook（CPU）&nbsp;|&nbsp; **无需 GPU**

---

## 实验概述

图像/视频压缩是媒体互联网的基础设施：无论是网页图片、视频流媒体还是即时通讯中的图片传输，都需要压缩算法在「文件体积」与「视觉质量」之间取得平衡。JPEG 是已经使用了三十多年的经典压缩标准，而神经压缩（Neural Compression）是近年来兴起的新方向，用深度学习模型替代传统的固定变换算法。

本实验通过对同一张合成测试图施加不同程度的压缩，对比 JPEG 风格的块量化压缩与一种简化的神经压缩模拟之间的视觉差异，帮助你直观理解两种技术路线在「失真模式」上的本质区别。

> ⚠️ **重要声明**：本实验使用简化模拟（Simplified Conceptual Simulation），用降采样/上采样和块平均来演示神经压缩与 JPEG 的视觉差异原理，**并非真实的神经编码器**。真实的神经图像压缩（如 CompressAI、BPG、VVC 中的学习式工具）需要端到端训练的卷积网络或 Transformer，通过大规模数据学习率失真最优的变换方式，性能远优于本演示中的简化版本。本实验的目的是帮助你建立「两种失真模式有本质区别」的直觉，而非精确复现神经编码器的真实性能。

本实验对应课程中「神经压缩与下一代编解码」部分的核心内容。

## 学习目标

完成本实验后，你应该能够：

1. 描述 JPEG 的基本工作流程（分块 → DCT 变换 → 量化 → 熵编码），并解释量化步骤为何有损；
2. 解释 JPEG 的块效应（Blocking Artifacts）产生的根本原因；
3. 说明神经压缩与传统固定算法压缩在原理上的本质区别（学到的自适应变换 vs 固定变换）；
4. 从可视化结果中识别 JPEG 与神经压缩（模拟）在不同压缩程度下的失真模式差异；
5. 理解本实验模拟与真实神经编解码器之间的差距，避免将简化演示的结论直接套用到真实系统评价上。

## 背景与基本原理

### JPEG 的工作原理

JPEG 压缩的核心流程可以概括为四步：

1. **分块（Block Partitioning）**：将图像切分为一个个 8×8 的小块；
2. **DCT 变换（Discrete Cosine Transform）**：对每个 8×8 块做离散余弦变换，把空间域的像素值转换为频域系数——低频系数代表整体亮度/色彩趋势，高频系数代表细节纹理；
3. **量化（Quantization）**：这是**唯一的有损步骤**。低频系数通常保留得较精细，高频系数则被粗粒度量化甚至直接舍弃——因为人眼对高频细节的丢失相对不敏感，但也正因如此，压缩率越高、量化越粗，细节损失就越明显；
4. **熵编码（Entropy Coding）**：对量化后的系数做无损压缩（如霍夫曼编码），进一步压缩文件体积。

### 为什么 JPEG 会产生块效应？

由于 DCT 变换和量化是**逐块独立**进行的，相邻块在量化过程中产生的舍入误差彼此独立、互不参考。当量化步长较大（压缩率高）时，相邻块边界处的像素值会出现不连续的跳变，形成肉眼可见的「方格状」痕迹——这就是**块效应（Block Artifacts）**，是 JPEG 在低码率下最典型的失真特征。

### 神经压缩的基本思路

神经压缩用一个**端到端训练**的编码器-解码器网络替代固定的 DCT+量化流程：

- **编码器（Encoder）**：将图像压缩为一个低维的潜在表示（Latent Representation），这个表示不是人工设计的频域系数，而是网络在大规模数据上学到的、能高效捕捉图像内容的压缩表征；
- **解码器（Decoder）**：从潜在表示重建图像；
- **端到端优化**：整个网络针对「率失真联合目标（Rate-Distortion Objective）」直接优化——即同时最小化压缩后的比特数和重建图像与原图的失真，而不是像 JPEG 那样先固定变换方式再单独调量化参数。

**关键区别**：JPEG 的 DCT 变换对所有图像内容都套用同一套固定规则；神经编码器的变换方式是从数据中学到的，能够针对不同图像内容自适应地分配「压缩预算」（比如给关键区域分配更多比特，给背景平滑区域分配更少比特）。

## 实验设计

**测试图像**：合成灰度图（128×128），包含棋盘格图案（高频细节区域，最难压缩）、平滑渐变条带（低频区域，最容易压缩）、粗线条（边缘特征）。这三类内容分别对应 JPEG 最容易产生块效应、最不易失真、以及边缘保真度测试的典型场景。

**JPEG 模拟 `jpeg_simulate()`**：用不同大小的块内均值替换来近似 JPEG 量化效果——块越大（对应压缩率越高），块内细节丢失越彻底，块边界处的不连续也越明显。

**神经压缩模拟 `neural_compress_simple()`**：用「降采样 → 上采样（双线性插值）」近似神经编码器在高压缩率下的行为——不产生块状边界，而是整体变得模糊、但空间上保持连续。代码中还定义了一个真实的卷积自编码器 `ConvAutoencoder`（编码器降采样到 1/8 分辨率，解码器还原），但由于本实验未对其进行训练，实际对比使用的是更快速的降采样模拟。

**对比设计**：4 个压缩级别（`Lossless / Light / Medium / Heavy`）× 2 种方法（JPEG / Neural），构成 2×4 的对比网格。

**预期观察**：低压缩级别下两种方法差异很小；随着压缩程度提高，JPEG 会在块边界处出现明显的方格状痕迹（尤其在棋盘格区域），而神经压缩模拟会呈现整体模糊但空间连续的失真。

## 运行环境说明

| 项目 | 说明 |
|------|------|
| 运行环境 | Kaggle Notebook |
| 计算资源 | CPU（无需 GPU） |
| 网络访问 | 不需要（Internet Off） |
| 主要依赖 | PyTorch、NumPy、Matplotlib（Kaggle 已预装） |

直接点击「Run All」即可运行全部实验，无需上传数据或安装额外包。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import io

print("PyTorch 已就绪 ✅")

## 步骤一：生成测试图像

下面的代码用 `create_demo_image()` 构建一张 128×128 的合成测试图，特意设计了三类内容：

- **棋盘格（Checkerboard）**：黑白像素高频交替，代表压缩算法最难处理的高频细节区域；
- **渐变条带（Gradient）**：平滑变化的低频区域，通常压缩损失最小；
- **细线条（Line）**：模拟锐利边缘，用于观察边缘保真度。

这样的设计让我们能在同一张图上，同时观察两种压缩方法对「高频细节」「低频平滑区域」「锐利边缘」这三类内容的不同处理方式。

In [ ]:
# 生成演示用测试图像：包含高频和低频区域
def create_demo_image(size=128):
    img = np.zeros((size, size), dtype=np.float32)
    # 棋盘格（高频）
    for i in range(0, size, 8):
        for j in range(0, size, 8):
            if (i//8 + j//8) % 2 == 0:
                img[i:i+8, j:j+8] = 0.9
            else:
                img[i:i+8, j:j+8] = 0.1
    # 平滑渐变（低频）
    for i in range(20, 60):
        img[i+60, 20:100] = 0.3 + (i-20)/40 * 0.6
    # 细线
    img[100:110, :] = 1.0
    return img

original = create_demo_image(128)

plt.figure(figsize=(4, 4))
plt.imshow(original, cmap='gray', vmin=0, vmax=1)
plt.title('Test Image\n（Checkerboard=HF|Gradient=LF|Line=Edge）')
plt.axis('off')
plt.show()

## 步骤二：定义 JPEG 与神经压缩的模拟实现

这部分代码定义了两个压缩模拟函数：

**① 神经压缩模拟（`ConvAutoencoder` + `neural_compress_simple`）**：`ConvAutoencoder` 是一个真实的卷积自编码器结构（3 层卷积编码器 + 3 层反卷积解码器），展示了神经压缩网络的典型结构；但由于训练需要额外时间，本实验实际使用 `neural_compress_simple()` 通过「降采样 + 双线性上采样」来快速模拟神经压缩在不同压缩率下的失真效果——这是一种简化近似，视觉效果与真实神经编码器类似（整体模糊但结构连续），但底层原理不同。

**② JPEG 模拟（`jpeg_simulate`）**：通过对不同大小的像素块取均值来近似 JPEG 量化后的效果——块越大，对应的压缩率越高，块内所有像素被替换为同一个均值，制造出块状的失真痕迹。

> 提醒：两个函数都在代码注释中明确标注了「这是简化模拟」，实际生产级压缩算法要复杂得多。

In [ ]:
# 简单的神经自编码器（卷积版本）
class ConvAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        # 编码器：到 1/4 尺寸
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 8, 3, stride=2, padding=1), nn.ReLU(),
        )
        # 解码器：恢复到原始尺寸
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(8, 32, 3, stride=2, padding=1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 3, stride=2, padding=1, output_padding=1), nn.Sigmoid(),
        )
    
    def forward(self, x):
        return self.decoder(self.encoder(x))

# 创建并训练一个小模型来演示概念（实际用预训练模型更快）
# 这里用简单方式演示：用不同程度的降采样来模拟"Neural"
# 这样无需训练，直接展示概念

def neural_compress_simple(img_tensor, compression_level):
    """
    简化模拟Neural：通过降采样+上采样模拟
    实际Neural比这复杂得多，但视觉效果类似
    """
    factors = [1, 2, 4, 8]
    factor = factors[min(compression_level, 3)]
    if factor == 1:
        return img_tensor.clone()
    
    _, _, h, w = img_tensor.shape
    # 降采样
    small = F.interpolate(img_tensor, size=(h//factor, w//factor), mode='area')
    # 上采样
    restored = F.interpolate(small, size=(h, w), mode='bilinear', align_corners=False)
    return restored

print("Neural模拟器就绪 ✅")
print("注：实际Neural用端到端训练的网络，这里用采样模拟原理展示")

In [ ]:
# JPEG（量化造成的方块效应）
def jpeg_simulate(img_tensor, quality):
    """
    模拟 JPEG 效果：
    - 对 8x8 块做平均（模拟量化）
    - quality 越低，块越大
    """
    block_sizes = [1, 4, 8, 16]
    block = block_sizes[min(quality, 3)]
    if block == 1:
        return img_tensor.clone()
    
    result = img_tensor.clone()
    _, _, h, w = result.shape
    for i in range(0, h, block):
        for j in range(0, w, block):
            i_end = min(i + block, h)
            j_end = min(j + block, w)
            result[:, :, i:i_end, j:j_end] = result[:, :, i:i_end, j:j_end].mean()
    return result

print("JPEG器就绪 ✅")
print("注：真实 JPEG 的块效应来自 8x8 DCT 量化，这里用块平均近似展示")

## 步骤三：对比可视化——JPEG vs 神经压缩

下方代码构建一个 2×4 的对比网格：上排为 JPEG 在 4 个压缩级别下的结果，下排为神经压缩模拟在相同压缩级别下的结果。

**观察重点**：
- 在 `Lossless` 列，两种方法应该几乎没有差异；
- 在 `Medium` 列开始，JPEG 的棋盘格区域会出现明显的方块边界，而神经压缩模拟呈现的是整体模糊但边界连续；
- 在 `Heavy` 列，JPEG 的失真非常剧烈且棱角分明，神经压缩模拟则表现为大范围模糊但仍保留了图案的大致轮廓。

In [ ]:
# 构建对比矩阵
img_t = torch.tensor(original).unsqueeze(0).unsqueeze(0).float()

levels = {'Lossless': 0, 'Light': 1, 'Medium': 2, 'Heavy': 3}

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for j, (label, lv) in enumerate(levels.items()):
    # JPEG
    jpeg_result = jpeg_simulate(img_t, lv)
    axes[0, j].imshow(jpeg_result.squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[0, j].set_title(f'JPEG\n{label}', fontsize=11)
    axes[0, j].axis('off')
    
    # Neural模拟
    neural_result = neural_compress_simple(img_t, lv)
    axes[1, j].imshow(neural_result.squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[1, j].set_title(f'Neural\n{label}', fontsize=11)
    axes[1, j].axis('off')

plt.suptitle('JPEG vs Neural Compression\n（Top: JPEG | Bottom: Neural）',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nKey observations:")
print("1. JPEG: visible blocks at medium+ compression (8x8 boundaries), checkerboard lost")
print("2. Neural: preserves structure but details blur (downsample-upsample smoothing)")
print("3. Real neural compression uses end-to-end trained CNN/Transformer, much better than this simulation")

## 实验结果与分析

运行实验后，你通常会观察到：

- **`Lossless` 列**：JPEG 和神经压缩模拟的结果与原图基本一致，几乎看不出差异；
- **`Light` 列**：两者都出现轻微失真，但差异尚不明显；
- **`Medium` 列**：JPEG 在棋盘格区域出现清晰可辨的方块边界（Block Artifacts），失真呈现「棱角分明」的特征；神经压缩模拟则呈现整体性的模糊，图案变得柔和但没有生硬的边界；
- **`Heavy` 列**：JPEG 的方块效应进一步加剧，棋盘格图案几乎完全被块状痕迹取代；神经压缩模拟则在保持结构连续性的同时严重丢失细节，看起来像被「打了马赛克后又做了模糊处理」。

### 现象解读

这组对比直观体现了两种失真模式的本质区别：**JPEG 的失真是「局部的、不连续的」**（逐块独立处理导致的边界断裂），**神经压缩（即使是本实验的简化模拟）的失真是「全局的、连续的」**（整体信息被压缩后再插值恢复，不会产生生硬边界）。这也是为什么真实的神经编码器往往能在相同压缩率下获得比 JPEG 更高的主观质量评分——它们避免了令人不悦的块效应，即使损失了一些细节，观感仍然更「自然」。

## 从实验到实际系统

本实验用简化模拟展示了核心失真差异，但真实的神经压缩系统要复杂得多：

- **真实神经编解码器**：如学术界的 Ballé et al. 系列工作、工业界的 CompressAI 框架，通常包含超先验网络（Hyperprior）、上下文模型（Context Model）等组件，通过端到端训练在大规模自然图像数据集上学习最优的率失真权衡；
- **新一代视频编码标准**：VVC（H.266）、AV1 等标准也在探索将神经网络模块（如神经网络环路滤波器）融入传统编解码框架，形成「混合编码」方案；
- **训练开销与推理成本**：神经编解码器需要 GPU 加速推理，相比 JPEG 的极低计算开销，实际部署需要权衡压缩效果与计算成本；
- **本实验 vs 真实系统的差距**：本实验的「降采样-上采样」模拟仅能演示「失真是否连续」这一个维度，无法体现真实神经编码器在语义保真度、码率自适应分配等方面的优势。

---

## 本实验小结

通过本实验，你应该掌握以下核心结论：

1. **JPEG 基于固定的分块 DCT 变换**：所有图像内容套用同一套频域变换和量化规则；
2. **量化是 JPEG 唯一的有损步骤**：压缩率越高，高频细节被量化得越粗糙；
3. **块效应源于逐块独立处理**：相邻块边界处的量化误差互不参考，导致视觉上的不连续；
4. **神经压缩使用学到的自适应变换**：能够根据图像内容动态分配压缩预算，失真模式呈现全局连续而非局部断裂；
5. **本实验是简化的概念演示**：真实神经编解码器的性能和失真特征远比本实验的降采样模拟复杂和优越。

---

## 思考与拓展

以下问题没有唯一答案，鼓励你修改代码并重新运行：

1. **换用自然照片**：将棋盘格测试图替换为一张真实照片，重新对比 JPEG 与神经压缩模拟在自然纹理上的失真差异。
2. **训练 `ConvAutoencoder`**：代码中已定义但未训练的卷积自编码器，尝试用简单的数据（如本实验的合成图案做数据增强）训练它，并与降采样模拟版本的效果做对比。
3. **绘制率失真曲线**：为两种方法各自计算压缩比与 PSNR/SSIM（复用实验三的实现）的关系曲线，量化比较两者在相同压缩比下的客观质量差异。
4. **调整块大小参数**：修改 `jpeg_simulate()` 中的 `block_sizes` 列表，观察块大小与失真严重程度的关系是否符合预期。
5. **对比降采样倍数**：修改 `neural_compress_simple()` 中的 `factors` 列表，观察降采样倍数增加时，模拟结果与真实神经压缩失真模式的相似程度如何变化。

---

← [实验三：视频质量评估 PSNR vs SSIM](https://www.kaggle.com/code/guopingtan/fmi-demo3-quality-assessment) &nbsp;|&nbsp; 🏠 [课程主页 · Course Home](https://www.kaggle.com/code/guopingtan/fmi-course-kaggle-hands-on-lab-start-here) &nbsp;|&nbsp; [实验五：语义通信 vs 传统传输 →](https://www.kaggle.com/code/guopingtan/fmi-demo5-semantic-communication)

**FMI Course · Kaggle Hands-on Lab** &nbsp;|&nbsp; MV-AI Lab · Hohai University